# Train/Test split
---
0.7/0.15/0.15

### Inladen
---
Bestanden en imports

In [1]:
from pathlib import Path
import pandas as pd

#Projectmap zoeken, werkt vanuit elke submap van models/
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "prepared_data").exists())
DATA_DIR = ROOT / "prepared_data"
DATA_FILE = DATA_DIR / "forecast_targets_with_weather.parquet"
SPLIT_FILE = DATA_DIR / "model_split.parquet"

In [2]:
#70% wordt gebruikt voor training, 15% voor validatie en 15% voor test
TRAIN_FRAC = 0.70
VAL_FRAC = 0.15
TEST_FRAC = 0.15

In [3]:
TARGETS = [("huis", "consumption"), ("huis", "generation"), ("carport", "consumption"), ("carport", "generation")]

### Targets naast elkaar zetten
---
- Is per 5 minuten- timestamp de `pow_imputed` beschikbaar voor alle vier de targets?

In [4]:
data = pd.read_parquet(DATA_FILE)
data["timestamp"] = pd.to_datetime(data["timestamp"])
targets = data[data[["location", "target_type"]].apply(tuple, axis=1).isin(TARGETS)].copy()

pow_per_target = targets.pivot_table(index="timestamp", columns=["location", "target_type"], values="pow_imputed", aggfunc="first").sort_index()

pow_per_target = pow_per_target.reindex(columns=pd.MultiIndex.from_tuples(TARGETS, names=["location", "target_type"]))

display(pow_per_target.isna().sum().to_frame("aantal missend"))

aantal missend
location target_type                
huis     consumption           27649
         generation            28044
carport  consumption           42449
         generation            45349

### Langste keten
---
- Langste keten `pow_imputed` selecteren

In [5]:
volledige_tijdlijn = pd.date_range(start=pow_per_target.index.min(), end=pow_per_target.index.max(), freq="5min")
pow_per_target = pow_per_target.reindex(volledige_tijdlijn)
pow_per_target.index.name = "timestamp"

alle_beschikbaar = pow_per_target.notna().all(axis=1)

blok = alle_beschikbaar.ne(alle_beschikbaar.shift()).cumsum()

geldige_blokken = blok[alle_beschikbaar]

beste_blok = geldige_blokken.value_counts().idxmax()

gezamenlijke_index = pow_per_target.index[(blok == beste_blok) & alle_beschikbaar]

common_start = gezamenlijke_index.min().ceil("1h")
common_end = (gezamenlijke_index.max() + pd.Timedelta(minutes=5)).floor("1h")

print("Gemeenschappelijke periode:", common_start, "tot en met", common_end)
print("Duur:", common_end - common_start)

Gemeenschappelijke periode: 2025-10-21 22:00:00+02:00 tot en met 2026-04-13 12:00:00+02:00
Duur: 173 days 14:00:00


### 70/15/15 split
---

In [6]:
tijdlijn = pd.date_range(start=common_start, end=common_end - pd.Timedelta(minutes=5), freq="5min")

n = len(tijdlijn)

i_train = int(n * TRAIN_FRAC)
i_val = int(n * (TRAIN_FRAC + VAL_FRAC))

train_start = tijdlijn[0]
train_end = tijdlijn[i_train].ceil("1h")

val_start = train_end
val_end = tijdlijn[i_val].ceil("1h")

test_start = val_end
test_end = common_end

print("TRAIN     :", train_start, "t/m", train_end, "aantal dagen:", train_end - train_start)
print("VALIDATION:", val_start, "t/m", val_end, "aantal dagen:", val_end - val_start)
print("TEST      :", test_start, "t/m", test_end, "aantal dagen:", test_end - test_start)

TRAIN     : 2025-10-21 22:00:00+02:00 t/m 2026-02-20 10:00:00+01:00 aantal dagen: 121 days 13:00:00
VALIDATION: 2026-02-20 10:00:00+01:00 t/m 2026-03-18 11:00:00+01:00 aantal dagen: 26 days 01:00:00
TEST      : 2026-03-18 11:00:00+01:00 t/m 2026-04-13 12:00:00+02:00 aantal dagen: 26 days 00:00:00


In [7]:
controle = pow_per_target.loc[(pow_per_target.index >= common_start) & (pow_per_target.index < common_end)]
controle.isna().sum()


location  target_type
huis      consumption    0
          generation     0
carport   consumption    0
          generation     0
dtype: int64

### Opslaan
---

In [8]:
split = pd.DataFrame([{
    "train_start": train_start,
    "train_end": train_end,
    "val_start": val_start,
    "val_end": val_end,
    "test_start": test_start,
    "test_end": test_end,
    "train_frac": TRAIN_FRAC,
    "val_frac": VAL_FRAC,
    "test_frac": TEST_FRAC,
}])

split.to_parquet(SPLIT_FILE, index=False)

split

,train_start,train_end,val_start,val_end,test_start,test_end,train_frac,val_frac,test_frac
0,2025-10-21 22:00:00+02:00,2026-02-20 10:00:00+01:00,2026-02-20 10:00:00+01:00,2026-03-18 11:00:00+01:00,2026-03-18 11:00:00+01:00,2026-04-13 12:00:00+02:00,0.7,0.15,0.15
